<a href="https://colab.research.google.com/github/Ivory-Uche/Portfolio-Pilot/blob/main/Portfolio_pilot_M3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PortfolioPilot - Milestone 3: cleaning, EDA, dimension reduction, feature set



**Run the cells in order, top to bottom.** Each cell uses variables made by the cells before it.
After a kernel restart, start again from cell 1.

Rule for M3: the test sets are created in Step 3 and then never explored. Every statistic, cap,
table and figure uses **training data only**.

In [3]:
%pip install numpy pandas scipy scikit-learn matplotlib yfinance

## Step 1 - Setting up

In [10]:
from google.colab import drive
drive.mount("/content/drive")          # a pop-up asks you to allow access: click Allow

import os
os.chdir("/content/drive/MyDrive/Portfolio_pilot/data/raw")
print("Working in:", os.getcwd())
print("Files here:", os.listdir())

from pathlib import Path
SCF_FILE = Path("SCFP2025.csv")
OUT_DIR  = Path("outputs")


SEED = 7275

# SCF column names in the summary extract (check against the SCF codebook)
COLS = {
    "household_id": "YY1", "record_id": "Y1", "weight": "WGT", "age": "AGE",
    "income": "INCOME", "networth": "NETWORTH", "liq": "LIQ", "debt": "DEBT",
    "equity": "EQUITY", "fin": "FIN", "yes_risk": "YESFINRISK", "no_risk": "NOFINRISK",
}
IMPLICATE = 1
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}
CAP_QUANTILE = 0.99            # caps for ratio features, from training data

K_RANGE = [2, 3, 4, 5, 6]
CLUSTER_METHOD = "kmeans"      # "kmeans" or "gmm"
CHOSEN_K = None                # None = automatic (best silhouette); or set a number after Step 9
MIN_CLUSTER_SHARE = 0.05
N_BOOTSTRAP = 20               # use 3 for a quick test, 20 for final results
PCA_THRESHOLD = 0.80

TICKERS = ["SPY", "VEA", "VWO", "AGG", "TLT", "TIP", "VNQ", "GLD", "DBC"]
ETF_START, ETF_END = "2016-01-01", "2026-01-01"   # end is exclusive: data through 2025-12-31
ETF_TRAIN_END, ETF_VAL_END = "2021-12-31", "2023-12-31"
ETF_FILE = OUT_DIR / "raw" / "etf_prices_raw.csv"

FIG_DIR, TAB_DIR, DATA_DIR = OUT_DIR / "figures", OUT_DIR / "tables", OUT_DIR / "processed"
for d in [FIG_DIR, TAB_DIR, DATA_DIR, ETF_FILE.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("SCF file found:", SCF_FILE.exists())
print("Outputs go to:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/Portfolio_pilot/data/raw
Files here: ['SCFP2025.csv', 'PortfolioPilot_ETF_Prices.csv', 'outputs']
SCF file found: True
Outputs go to: outputs


## Step 2 - Import and helpers

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(SEED)

NUMBERS = {}   # every count quoted in the report is stored here and saved in the last step
def note(section, key, value):
    NUMBERS.setdefault(section, {})[key] = value

def to_plain(v):
    """Make numpy values JSON-friendly."""
    if isinstance(v, dict):
        return {str(k): to_plain(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [to_plain(x) for x in v]
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return round(float(v), 6)
    return v

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def weighted_resample(df, seed):
    """Draw households with probability proportional to the survey weight."""
    rng = np.random.default_rng(seed)
    p = df["weight"].to_numpy(float)
    return df.iloc[rng.choice(len(df), size=len(df), replace=True, p=p / p.sum())]

def weighted_median(x, w):
    o = np.argsort(np.asarray(x))
    x, w = np.asarray(x, float)[o], np.asarray(w, float)[o]
    c = np.cumsum(w)
    return float(x[np.searchsorted(c, 0.5 * c[-1])])

print("Ready")

Ready
